# AgriSense - Data Visualization: Histograms &amp; Boxplots

Welcome to Section 4.40! In this notebook, we will learn how to visualize data distributions using **Histograms** and **Boxplots**. 

Before we send data to the React frontend or train Machine Learning models, we *must* understand its shape. Are there outliers? Is the price skewed? Visualizations answer these questions instantly.

### What we will learn:
1. **Histograms:** To see the frequency distribution of a single variable (e.g., Prices, Yields).
2. **Boxplots:** To compare distributions across categories (e.g., Prices by Crop, Prices by State) and spot outliers.
3. **Saving Plots:** How to save our charts for reports or dashboards.

In [ ]:
# Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os
import warnings

# Suppress minor warnings for cleaner output
warnings.filterwarnings('ignore')

# Create output directory for saving figures if it doesn't exist
Path("../outputs/figures").mkdir(parents=True, exist_ok=True)

# Set a clean style for our seaborn plots
sns.set_theme(style="whitegrid")
print("Libraries imported successfully!")

## 1. Load the Cleaned Data
We will load our `mandi_prices.csv` dataset. For demonstration on Yield distributions, we will simulate a yield column since our primary raw dataset is focused entirely on pricing info.

In [ ]:
# Load data using pathlib for cross-platform safety
data_path = Path("../data/raw/mandi_prices.csv")
df = pd.read_csv(data_path)

# Clean column names (Applying what we learned in Section 4.36)
df.columns = df.columns.str.lower().str.strip()

# If the file doesn't have a 'yield_hectare' column, let's artificially create one for demonstration
import numpy as np
if 'yield_hectare' not in df.columns:
    np.random.seed(42)
    # Simulate yield between 10 and 50 quintals per hectare, centered around 30
    df['yield_hectare'] = np.random.normal(loc=30, scale=8, size=len(df))

print(f"Data loaded successfully! Shape: {df.shape}")
display(df.head(3))

## 2. Histograms: Modal Price Distribution
A histogram groups data into "bins" and shows how many records fall into each bin.

In [ ]:
# Set the figure size
plt.figure(figsize=(10, 6))

# Create the histogram for modal_price
sns.histplot(data=df, x='modal_price', bins=15, kde=True, color='skyblue')

# Add descriptions and styling
plt.title('Distribution of Modal Prices', fontsize=16)
plt.xlabel('Modal Price (₹)', fontsize=12)
plt.ylabel('Frequency / Number of Records', fontsize=12)

# Save the figure to our outputs folder
plt.savefig('../outputs/figures/price_histogram.png', dpi=300, bbox_inches='tight')
plt.show()

### 💡 Insights: Modal Price Distribution
* The KDE (Kernel Density Estimate - the line curve) shows the general shape of the distribution.
* You can observe where most of the prices cluster (the peak).
* If there is a long "tail" to the right, it means there are a few unusually high prices skewing the data. We visualize this beforehand so the API knows what standard ranges to emit!

## 3. Histograms: Yield Distribution
Now let's look at the yield column (quintals per hectare).

In [ ]:
plt.figure(figsize=(10, 6))

# Create a histogram for our yield data
sns.histplot(data=df, x='yield_hectare', bins=15, kde=True, color='lightgreen')

plt.title('Distribution of Crop Yield (Quintals/Hectare)', fontsize=16)
plt.xlabel('Yield (q/ha)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)

plt.savefig('../outputs/figures/yield_histogram.png', dpi=300, bbox_inches='tight')
plt.show()

### 💡 Insights: Yield Distribution
* Crop yield tends to follow a **normal (bell-shaped) distribution**, unlike market prices which can be highly sporadic.
* Most simulated farms reported a yield around the 30 q/ha mark (our median).
* Understanding this curve helps set standard benchmarks for the Yield Prediction Machine Learning models later.

## 4. Boxplots: Comparing Price by Crop
Boxplots are incredible for showing the **Median**, **Quartiles**, and **Outliers** all at once inside a tight package.

In [ ]:
plt.figure(figsize=(10, 6))

# Create a boxplot comparing prices across different commodities
sns.boxplot(data=df, x='commodity', y='modal_price', hue='commodity', palette='Set2', legend=False)

plt.title('Modal Price Distribution by Crop', fontsize=16)
plt.xlabel('Crop (Commodity)', fontsize=12)
plt.ylabel('Modal Price (₹)', fontsize=12)

plt.savefig('../outputs/figures/price_by_crop_boxplot.png', dpi=300, bbox_inches='tight')
plt.show()

### 💡 Insights: Price by Crop
* **Median Price:** The solid line inside the colored box represents the median price. Notice how Rice is visibly more expensive than Potato.
* **Price Range (IQR):** The "box" represents the middle 50% of the data. Rice prices have a noticeably wider distribution than Wheat.
* **Consistency:** Most wheat prices fall tightly between ₹2200 - ₹2600. It's a highly stable market!
* **Outliers:** Depending on the slice of data, dots rendering outside the "whiskers" mark extreme outliers. You might notice some high outliers in Tomato prices due to sudden seasonal shortages.

## 5. Boxplots: Comparing Price by State
Let's see if geographical region drastically affects the price variance. This is highly requested data on the frontend AgriSense dashboards!

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(data=df, x='state', y='modal_price', hue='state', palette='Pastel1', legend=False)

plt.title('Modal Price Distribution by State', fontsize=16)
plt.xlabel('State', fontsize=12)
plt.ylabel('Modal Price (₹)', fontsize=12)
plt.xticks(rotation=45) # Rotate long state labels

plt.savefig('../outputs/figures/price_by_state_boxplot.png', dpi=300, bbox_inches='tight')
plt.show()

### 💡 Insights: Price by State
* States with a much "taller" box represent high market volatility – Prices fluctuate wildly.
* States with a compressed box represent prices that are strictly regulated or highly consistent.
* This geographic pricing insight will power the state-by-state filter menus inside `app/market/page.tsx` on the frontend UI.

## 6. Key Findings &amp; Conclusion

### Why do we visualize before building APIs?
Visualizing distributions prior to shipping apps prevents disaster. 

1. **Frontend App Resilience:** On the Market Dashboard UI, we now know exactly what the standard scale of our Y-axis should be for our Recharts/Plotly bar charts. Because we identified outliers via boxplots, we know our frontend needs to handle extreme upper bounds gracefully without breaking the layout.
2. **API Data Design:** Knowing the accurate distribution helps us write validation logic inside FastAPI (e.g., throwing a warning if entered modal price exceeds expected boxplot quartiles).
3. **Machine Learning Accuracy:** The `scikit-learn` algorithms we build for Yield/Price prediction will necessitate outlier handling (perhaps clipping them) before they begin training to guarantee top accuracy.

### Next Steps:
Now that we deeply understand the visual shape (frequency &amp; spread) of the data, the next section will tackle **Correlation Analysis**. We will determine how drastically features like Rainfall and Temperature visually correlate to these prices and yields!